In [14]:
!pip install openai -q

In [15]:
!pip install -q openai pdfplumber gradio numpy

In [16]:
import pdfplumber, re

def extract_clauses(file):
    """
    Extracts about 5–10 clauses from a PDF or TXT contract.
    """
    text = ""
    if file.name.endswith(".pdf"):
        with pdfplumber.open(file.name) as pdf:
            for page in pdf.pages:
                page_text = page.extract_text()
                if page_text:
                    text += page_text + "\n"
    else:
        text = file.read().decode("utf-8", errors="ignore")

    # simple heuristic split: numbered items or capitalized headings
    chunks = re.split(r'(\n\d+\.\s+|\n[A-Z][A-Z\s]+:)', text)
    clauses = [c.strip() for c in chunks if len(c.strip()) > 60]

    print(f"✅ Extracted {len(clauses)} clauses (showing first 3):\n")
    for c in clauses[:3]:
        print("-", c[:200].replace("\n", " ") + "...\n")

    return clauses[:10]   # limit for demo


In [33]:
from google.colab import files
uploaded = files.upload()          # choose a small txt or pdf
sample_file = next(iter(uploaded.items()))
open(sample_file[0], "wb").write(sample_file[1])
fake = type("f", (), {"name": sample_file[0], "read": lambda: open(sample_file[0],"rb").read()})
extract_clauses(fake)


Saving contract_4.pdf to contract_4.pdf
✅ Extracted 1 clauses (showing first 3):

- Contract #4 This Agreement ("Agreement") is made between "Company" and "User" on the Effective Date. Clause 1 – General The User accepts full responsibility for any taxes, fines or penalties arising f...



['Contract #4\nThis Agreement ("Agreement") is made between "Company" and "User" on the Effective Date.\nClause 1 – General\nThe User accepts full responsibility for any taxes, fines or penalties arising from use of the Services.\nClause 2 – General\nEither Party may propose amendments in writing with mutual consent.\nClause 3 – General\nThe User expressly waives any right to seek injunctive or equitable relief against the Company.\nClause 4 – General\nThe Parties agree to communicate electronically via email for all formal notices.\nClause 5 – General\nThe Company reserves the right to modify the terms without explicit user consent and such modifications\nwill be effective immediately upon posting.\nClause 6 – General\nThis Agreement shall be governed by the laws of California, USA.\nClause 7 – General\nThe Company may share Customer data with third parties for any purpose it deems necessary without\nprior notice.\nClause 8 – Liability\nAny limitation of liability includes liability f

In [18]:

from openai import OpenAI

# 🔑 Replace this with your NVIDIA API key from build.nvidia.com
client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key="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"
)


In [34]:
!pip install -q faiss-cpu rank_bm25 numpy

import numpy as np, faiss, math
from rank_bm25 import BM25Okapi

# ---------- 1) Safe clause library (expandable) ----------
SAFE_TEMPLATES = [
  {"topic":"confidentiality", "text":"Each party shall keep all confidential information private and use it solely for contractual purposes."},
  {"topic":"termination",     "text":"Either party may terminate this agreement with thirty (30) days' written notice for reasonable cause."},
  {"topic":"liability",       "text":"Liability is limited to direct damages and shall not exceed the total contract value paid under this agreement."},
  {"topic":"indemnity",       "text":"Each party shall indemnify and hold the other harmless for losses arising from its own breach or negligence."},
  {"topic":"governing_law",   "text":"This agreement shall be governed by and construed in accordance with the laws of California, without regard to conflicts of law."},
  {"topic":"payment_terms",   "text":"Invoices are due within thirty (30) days of receipt; late payments may incur interest at the lesser of 1.5% per month or the maximum allowed by law."},
  {"topic":"ip_ownership",    "text":"Each party retains ownership of its pre-existing IP. Deliverables created under this agreement are owned by Company upon full payment, subject to licensor’s background IP."},
  {"topic":"data_protection", "text":"Parties will implement reasonable administrative, technical, and physical safeguards to protect personal data and will process such data only for the purposes of this agreement."},
  {"topic":"subcontracting",  "text":"Contractor may not subcontract material obligations without prior written consent; Contractor remains fully responsible for subcontractors' performance."},
  {"topic":"warranty",        "text":"Contractor warrants that services will be performed in a professional and workmanlike manner in accordance with industry standards for ninety (90) days."}
]

CORPUS = [d["text"] for d in SAFE_TEMPLATES]
TOPICS = [d["topic"] for d in SAFE_TEMPLATES]

# ---------- 2) Embeddings index (NVIDIA retriever model) ----------
def get_embedding(text, input_type="passage"):
    emb = client.embeddings.create(
        input=[text],
        model="nvidia/llama-3.2-nemoretriever-300m-embed-v2",
        encoding_format="float",
        extra_body={"input_type": input_type, "truncate": "NONE"}
    )
    return np.array(emb.data[0].embedding, dtype="float32")

EMB_DIM = len(get_embedding("warm start"))  # probe once
DOC_EMBS = np.vstack([get_embedding(t, "passage") for t in CORPUS])

faiss_index = faiss.IndexFlatIP(EMB_DIM)
# normalize for cosine via inner product
DOC_EMBS_N = DOC_EMBS / np.linalg.norm(DOC_EMBS, axis=1, keepdims=True)
faiss_index.add(DOC_EMBS_N)

# ---------- 3) BM25 index ----------
tokenized = [c.lower().split() for c in CORPUS]
bm25 = BM25Okapi(tokenized)

# ---------- 4) Hybrid retrieval + MMR diversification ----------
def hybrid_search(query_text, top_k=8, alpha=0.6):
    # embeddings
    q = get_embedding(query_text, "query")
    qn = q / np.linalg.norm(q)
    sim, idx = faiss_index.search(qn.reshape(1,-1), min(top_k, len(CORPUS)))
    emb_scores = sim[0].tolist()
    emb_idx = idx[0].tolist()

    # BM25
    bm_scores = bm25.get_scores(query_text.lower().split()).tolist()

    # combine (normalize both to 0–1, weighted by alpha)
    def norm(arr):
        a = np.array(arr, dtype=float)
        return (a - a.min()) / (a.max() - a.min() + 1e-9)
    emb_norm = norm([emb_scores[emb_idx.index(i)] if i in emb_idx else 0 for i in range(len(CORPUS))])
    bm_norm  = norm(bm_scores)
    hybrid = alpha*emb_norm + (1-alpha)*bm_norm

    # pick top candidates then do MMR for diversity
    top_indices = np.argsort(hybrid)[::-1][:top_k].tolist()
    selected, selected_vecs = [], []
    lamb = 0.75
    while top_indices and len(selected) < min(5, top_k):  # final shortlist
        best_i, best_score = None, -1
        for i in top_indices:
            rel = hybrid[i]
            div = 0 if not selected_vecs else max(
                np.dot(DOC_EMBS_N[i], v) for v in selected_vecs
            )
            mmr_score = lamb*rel - (1-lamb)*div
            if mmr_score > best_score:
                best_score, best_i = mmr_score, i
        selected.append(best_i)
        selected_vecs.append(DOC_EMBS_N[best_i])
        top_indices.remove(best_i)

    candidates = [{"topic": TOPICS[i], "text": CORPUS[i], "score": float(hybrid[i])} for i in selected]
    return sorted(candidates, key=lambda x: x["score"], reverse=True)

# ---------- 5) LLM re-rank (49B cross-check for legal fit) ----------
def rerank_with_llm(user_clause, candidates):
    prompt = (
      "Rank the following reference clauses by how well they match the user's clause intent and provide a brief reason. "
      "Return JSON with fields: ranked (array of indices best→worst), reasons (array of strings).\n\n"
      f"User Clause:\n{user_clause}\n\n"
      "Candidates:\n" +
      "\n".join([f"[{i}] Topic: {c['topic']}\nText: {c['text']}\n" for i,c in enumerate(candidates)])
    )
    resp = call_nemotron(prompt, model="nvidia/llama-3.3-nemotron-super-49b-v1.5", max_tokens=10000)
    # minimal parse fallback: pick the first candidate if parsing fails
    import json, re
    try:
        js = json.loads(re.search(r'\{.*\}', resp, re.S).group(0))
        order = js.get("ranked", [])
        return candidates[order[0]] if order else candidates[0], resp
    except Exception:
        return candidates[0], resp

# ---------- 6) Main: retrieve_best_reference() ----------
def retrieve_best_reference(user_clause):
    cands = hybrid_search(user_clause, top_k=8, alpha=0.6)
    best, judge_expl = rerank_with_llm(user_clause, cands)
    return {"best": best, "candidates": cands, "judge_explanation": judge_expl}

In [35]:
def call_nemotron(prompt, model="nvidia/nemotron-4-340b-instruct", max_tokens=10000):
    """Helper function to call the Nemotron language model."""
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": prompt},
        ],
        max_tokens=max_tokens,
    )
    return response.choices[0].message.content

In [36]:
def analyze_and_remedy_clause(clause):
    """
    Compare clause vs safe reference, detect risks, and propose a rewrite.
    """
    # find the best-matching safe template
    result = retrieve_best_reference(clause)
    ref = result["best"]["text"]
    topic = result["best"]["topic"]

    # --- 1️⃣ Reasoner (Nemotron 49B) ---
    analysis_prompt = (
        f"You are a legal risk analysis expert. "
        f"Compare the following contract clause to the standard safe clause and identify all risks, vague terms, and unfair wording.\n\n"
        f"User Clause:\n{clause}\n\n"
        f"Reference Safe Clause ({topic}):\n{ref}\n\n"
        f"Return a concise bullet list of issues found, each starting with 🔴 if risky, 🟡 if unclear, or 🟢 if safe."
    )
    analysis = call_nemotron(
        analysis_prompt,
        model="nvidia/llama-3.3-nemotron-super-49b-v1.5",
        max_tokens=10000
    )

    # --- 2️⃣ Generator (Nemotron 9B) ---
    rewrite_prompt = (
        f"Rewrite the following clause to remove the risks mentioned below and make it balanced for both parties.\n\n"
        f"Clause:\n{clause}\n\n"
        f"Issues:\n{analysis}\n\n"
        f"Return only the improved clause."
    )
    rewrite = call_nemotron(
        rewrite_prompt,
        model="nvidia/nvidia-nemotron-nano-9b-v2",
        max_tokens=2048
    )

    return f"🧩 Clause:\n{clause}\n\n⚠️ Risk Analysis:\n{analysis}\n\n✅ Safer Rewrite:\n{rewrite}\n{'-'*90}"


In [37]:
# assuming you already created `file_obj` earlier
clauses = extract_clauses(fake)

for i, c in enumerate(clauses, 1):
    print(f"\n\n===== Clause {i} =====")
    print(analyze_and_remedy_clause(c))

✅ Extracted 1 clauses (showing first 3):

- Contract #4 This Agreement ("Agreement") is made between "Company" and "User" on the Effective Date. Clause 1 – General The User accepts full responsibility for any taxes, fines or penalties arising f...



===== Clause 1 =====
🧩 Clause:
Contract #4
This Agreement ("Agreement") is made between "Company" and "User" on the Effective Date.
Clause 1 – General
The User accepts full responsibility for any taxes, fines or penalties arising from use of the Services.
Clause 2 – General
Either Party may propose amendments in writing with mutual consent.
Clause 3 – General
The User expressly waives any right to seek injunctive or equitable relief against the Company.
Clause 4 – General
The Parties agree to communicate electronically via email for all formal notices.
Clause 5 – General
The Company reserves the right to modify the terms without explicit user consent and such modifications
will be effective immediately upon posting.
Clause 6 – General
This

In [ ]:
import gradio as gr
import time


def legal_lens_interface(file, progress=gr.Progress(track_tqdm=True)):
   """
   Upload a PDF or TXT -> extract clauses -> analyze + rewrite with visible progress.
   """
   print("🚀 File received:", file.name if file else "None")


   if not file:
       return "⚠️ Please upload a contract file."


   # Step 1: Extract clauses
   file_obj = type("f", (), {"name": file.name, "read": lambda: file.read()})
   clauses = extract_clauses(file_obj)


   if not clauses:
       return "⚠️ No valid clauses found in the document."


   output = []
   for i, c in enumerate(clauses, 1):
       progress(i / len(clauses), desc=f"Analyzing Clause {i}/{len(clauses)}...")
       try:
           start = time.time()
           result = analyze_and_remedy_clause(c)
           end = time.time()
           output.append(f"### Clause {i} (⏱️ {end-start:.1f}s)\n{result}")
       except Exception as e:
           output.append(f"⚠️ Error analyzing clause {i}: {e}")
   return "\n\n".join(output)




# ✅ Launch UI (Colab-safe)
ui = gr.Interface(
   fn=legal_lens_interface,
   inputs=gr.File(label="📄 Upload Contract (PDF or TXT)"),
   outputs="markdown",
   title="⚖️ LegalLens – AI Contract Risk Analyzer",
   description=(
       "Upload a legal contract to detect risky (🔴), unclear (🟡), and safe (🟢) clauses. "
       "Nemotron-49B performs risk reasoning, and Nemotron-9B generates safer rewrites."
   ),
   theme="default",
)


ui.launch(share=True, inline=True, debug=True, height=800)





Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://f57862e92e3e40bc55.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/protocols/http/h11_impl.py", line 403, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/middleware/proxy_headers.py", line 60, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/fastapi/applications.py", line 1133, in __call__
    await super().__call__(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/applications.py", line 113, in __call__
    await self.middleware_stack(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/errors.py", line 186, in __call__
    raise exc
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/errors.py",

🚀 File received: /tmp/gradio/c253371cbd81e80a14ed5ab8dd9853b7e2fa65fe82ade8c71a7324f4c04107ce/contract_10.pdf
✅ Extracted 1 clauses (showing first 3):

- Contract #10 This Agreement ("Agreement") is made between "Company" and "User" on the Effective Date. Clause 1 – General Delivery of products will be subject to availability and lead times announced i...



In [51]:
!pkill -f gradio
